# Hansen Ch.9 Hypothesis Testing — 计算

理论见 md。本 notebook：**9.25–9.29**。

In [ ]:

import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")

def ols_hc(y, X, df_corr=True):
    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    XXinv = np.linalg.inv(X.T @ X)
    h = np.sum(X * (X @ XXinv), axis=1)
    u = X * (e / np.clip(1 - h, 1e-12, None))[:, None]
    meat = u.T @ u
    if df_corr:
        meat = meat * (n / (n - k))
    V = XXinv @ meat @ XXinv
    return beta, e, V, n, k

def wald(R, beta, V, c=None):
    R = np.atleast_2d(R)
    if c is None:
        c = np.zeros(R.shape[0])
    d = R @ beta - c
    W = float(d.T @ np.linalg.inv(R @ V @ R.T) @ d)
    q = R.shape[0]
    return W, q, float(1 - stats.chi2.cdf(W, q))


## 9.25 Invest1993 (year=1987)

In [ ]:

inv = pd.read_excel(ROOT / "Invest1993/Invest1993.xlsx")
d = inv[inv.year == 1987][["inva", "vala", "cfa", "debta"]].apply(pd.to_numeric, errors="coerce").dropna()
y = d.inva.values
X = np.column_stack([d.vala, d.cfa, d.debta, np.ones(len(d))])
b, e, V, n, k = ols_hc(y, X)
print("n =", n)
print(pd.DataFrame({"beta": b, "SE": np.sqrt(np.diag(V)),
                    "lo": b - 1.96 * np.sqrt(np.diag(V)),
                    "hi": b + 1.96 * np.sqrt(np.diag(V))},
                   index=["Q", "C", "D", "int"]))
print("Wald C=D=0:", wald(np.array([[0,1,0,0],[0,0,1,0]], float), b, V))
print("Wald Q=0:", wald(np.array([[1,0,0,0]], float), b, V))
Q, C, D = d.vala.values, d.cfa.values, d.debta.values
Xq = np.column_stack([Q, C, D, Q**2, C**2, D**2, Q*C, Q*D, C*D, np.ones(len(d))])
bq, _, Vq, _, _ = ols_hc(y, Xq)
R6 = np.zeros((6, 10))
for i, j in enumerate([3, 4, 5, 6, 7, 8]):
    R6[i, j] = 1
print("Wald 6 nonlinear:", wald(R6, bq, Vq))


## 9.26 Nerlove1963

In [ ]:

ner = pd.read_excel(ROOT / "Nerlove1963/Nerlove1963.xlsx")
for c in ner.columns:
    ner[c] = pd.to_numeric(ner[c], errors="coerce")
ner = ner.dropna()
y = np.log(ner.Cost.values)
X = np.column_stack([np.ones(len(ner)), np.log(ner.output), np.log(ner.Plabor),
                     np.log(ner.Pcapital), np.log(ner.Pfuel)])
b, e, V, n, k = ols_hc(y, X)
print("n =", n)
print(pd.Series(b, index=["const", "logQ", "logPL", "logPK", "logPF"]))
print("SE:", np.sqrt(np.diag(V)))
print("Wald CRS:", wald(np.array([[0, 0, 1, 1, 1]], float), b, V, c=np.array([1.0])))


## 9.27 MRW1992

In [ ]:

mrw = pd.read_excel(ROOT / "MRW1992/MRW1992.xlsx")
m = mrw[mrw.N == 1]
y = (np.log(m.Y85) - np.log(m.Y60)).values
X = np.column_stack([
    np.log(m.Y60), np.log(m.invest / 100), np.log(m.pop_growth / 100 + 0.05),
    np.log(m.school / 100), np.ones(len(m))
])
b, e, V, n, k = ols_hc(y, X)
print("n =", n, "beta =", b)
print("Wald sum I+G+S = 0:", wald(np.array([[0, 1, 1, 1, 0]], float), b, V))


## 9.28–9.29 CPS marriage / education returns

In [ ]:

df = pd.read_excel(ROOT / "cps09mar/cps09mar.xlsx")
df["experience"] = df.age - df.education - 6
df["lwage"] = np.log(df.earnings / (df.hours * df.week))
df["exp2"] = (df.experience ** 2) / 100

black = df[(df.race == 2) & (df.hisp == 0)].copy().reset_index(drop=True)
for code, name in [(1, "m1"), (2, "m2"), (3, "m3"), (4, "wid"), (5, "div"), (6, "sep")]:
    black[name] = (black.marital == code).astype(float)
black["NE"] = (black.region == 1).astype(float)
black["South"] = (black.region == 3).astype(float)
black["West"] = (black.region == 4).astype(float)
y = black.lwage.to_numpy()
X = np.column_stack([
    black.education, black.experience, black.exp2, black.female,
    black.m1, black.m2, black.m3, black.wid, black.div, black.sep,
    black.NE, black.South, black.West, np.ones(len(black))
])
b, e, V, n, k = ols_hc(y, X)
R = np.zeros((6, 14))
for i, j in enumerate(range(4, 10)):
    R[i, j] = 1
print("9.28 n=", n, "Wald marriage=0:", wald(R, b, V))

sub = df[((df.race == 1) | (df.race == 2)) & (df.hisp == 0)].copy().reset_index(drop=True)
wm = ((sub.race == 1) & (sub.female == 0)).to_numpy().astype(float)
wf = ((sub.race == 1) & (sub.female == 1)).to_numpy().astype(float)
bm = ((sub.race == 2) & (sub.female == 0)).to_numpy().astype(float)
bf = ((sub.race == 2) & (sub.female == 1)).to_numpy().astype(float)
edu, exp, exp2 = sub.education.to_numpy(float), sub.experience.to_numpy(float), sub.exp2.to_numpy(float)
y = sub.lwage.to_numpy(float)
X = np.column_stack([edu * wm, edu * wf, edu * bm, edu * bf, exp, exp2, wf, bm, bf, np.ones(len(sub))])
b, e, V, n, k = ols_hc(y, X)
print("9.29 n=", n, "edu returns", b[:4])
R = np.array([[1, -1, 0, 0, 0, 0, 0, 0, 0, 0],
              [1, 0, -1, 0, 0, 0, 0, 0, 0, 0],
              [1, 0, 0, -1, 0, 0, 0, 0, 0, 0]], float)
print("9.29 Wald common return:", wald(R, b, V))
